## Loading Alita v3.03 weights into PyTorch and compiling to Torchscript

Things to note: 
- the model was trained on a GPU so we need to load weights and re-compile to CPU
- it's important to check what version of torchvision (if used here) and torch you're running in this notebook environment & be sure they match the versions pinned in the deployment container's Dockerfile
- The expected input is 480x480
- The model was trained on PyTorch Lightning, which structures checkpoints a bit differently than PyTorch, so there are few additional steps required to load the model properly.
- It's also using a custom classifier head, which is defined below.

In [1]:
import torch
import torch.nn as nn
import timm

/opt/homebrew/Caskroom/miniforge/base/envs/alitav3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# using a custom classifier head defined in AddaxAI:
# https://github.com/PetervanLunteren/AddaxAI/blob/main/classification_utils/model_types/alita-v3.03/classify_detections.py

class ClassifierHead(nn.Module):
    def __init__(self, num_features, num_classes, dropout_rate=0.2):
        super().__init__()
        self.linear = nn.Linear(num_features, num_features//2)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.output_layer = nn.Linear(num_features//2, num_classes)

    def forward(self, x):
        x = self.linear(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.output_layer(x)
        return x

In [ ]:
# create model and replace classifier head

model_name = "tf_efficientnetv2_s.in21k"
num_classes = 81
device = torch.device('cpu')

model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
in_features = model.classifier.in_features
print(f'There are {in_features} input features to the classifier head and {num_classes} outputs')
model.classifier = ClassifierHead(in_features, num_classes)

There are 1280 input features to the classifier head and 81 outputs


In [22]:
# load weights, stripping 'backbone.' prefix and skipping classifier weights

checkpoint_path = "./original-model/Exp_60_run_01_best_weights.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

new_state_dict = {}
for k, v in checkpoint.items():
    # Remove 'backbone.' prefix if present
    if k.startswith('backbone.'):
        k = k[9:]
    # Skip all classifier weights (linear and output_layer)
    if k.startswith('classifier.linear') or k.startswith('classifier.output_layer'):
        continue
    new_state_dict[k] = v

model.load_state_dict(new_state_dict, strict=False)
model.to(device)
model.eval()

EfficientNet(
  (conv_stem): Conv2dSame(3, 24, kernel_size=(3, 3), stride=(2, 2), bias=False)
  (bn1): BatchNormAct2d(
    24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): ConvBnAct(
        (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNormAct2d(
          24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (drop_path): Identity()
      )
      (1): ConvBnAct(
        (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNormAct2d(
          24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (drop_path):

In [12]:
# Test the model with a dummy input
dummy = torch.randn(1, 3, 480, 480)
out = model(dummy)
print(out.shape)

torch.Size([1, 81])


In [14]:
# Save out the whole model for future inference deployment
# https://pytorch.org/tutorials/beginner/saving_loading_models.html

compiled_path = './exported-model/alitav3_compiled_cpu.pt'

model_scripted = torch.jit.script(model) # Export to TorchScript
model_scripted.save(compiled_path) # Save